# AML Graph Autoencoder - End-to-End Pipeline
This notebook demonstrates the unified AML pipeline: Simulation -> Graph Construction -> GATv2 Training -> Anomaly Detection Verification.

In [ ]:
%matplotlib inline
import os, sys
sys.path.append(os.path.abspath("../"))
import yaml, torch, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from src.utils.reproducibility import set_seed, save_checkpoint, load_checkpoint, get_device
from src.data.generator import generate_transactions
from src.graph.builder import GraphBuilder
from src.models.gnn import AMLGraphAutoencoder, compute_combined_loss
from src.explain.interpreter import AnomalyInterpreter

# 0. Setup
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
set_seed(42)
device = get_device(config)

DATA_PATH = '../data/ledger.csv'
USE_EXISTING_DATA = False
MIN_EDGE_RISK_QUANTILE = 0.80

os.makedirs('../artifacts', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../data', exist_ok=True)

print(f"Setup complete. Using device: {device}")

## 1. Data Generation / Loading

In [ ]:
if USE_EXISTING_DATA and os.path.exists(DATA_PATH):
    print(f"Loading existing data from {DATA_PATH}...")
    full_tx_df = pd.read_csv(DATA_PATH)
    full_tx_df['date'] = pd.to_datetime(full_tx_df['date'])
else:
    print("Generating new synthetic data...")
    NUM_CUSTOMERS = config['data']['num_customers']
    full_tx_df = generate_transactions(num_customers=NUM_CUSTOMERS, num_days=40, anomaly_ratio=0.05)
    full_tx_df.to_csv(DATA_PATH, index=False)
    print(f"New data saved to {DATA_PATH}")

c_id_col = config['data']['column_mapping']['customer_id']

# Identify Typologies
pt_customers = full_tx_df[full_tx_df['label'] == 1][c_id_col].unique().tolist()
specific_pt_customers = full_tx_df[full_tx_df['label'] == 6][c_id_col].unique().tolist()
persistent_customers = full_tx_df[full_tx_df['label'] == 5][c_id_col].unique().tolist()

print(f"Pass-Through Customers: {pt_customers}")
print(f"Specific Pass-Through (900-1300): {specific_pt_customers}")

train_df = full_tx_df[full_tx_df['date'] < '2026-02-01'].copy()
oot_df = full_tx_df[full_tx_df['date'] >= '2026-02-01'].copy()

## 2. Graph Construction

In [ ]:
builder = GraphBuilder(config)
train_data = builder.build_graph(train_df)

node_scaler = StandardScaler()
edge_scaler = StandardScaler()
x_scaled = node_scaler.fit_transform(train_data.x.numpy())
edge_attr_scaled = edge_scaler.fit_transform(train_data.edge_attr.numpy())

train_data.x = torch.from_numpy(x_scaled).float().to(device)
train_data.edge_index = train_data.edge_index.to(device)
train_data.edge_attr = torch.from_numpy(edge_attr_scaled).float().to(device)

print(f"Training Graph: {train_data.num_nodes} nodes, {train_data.num_edges} edges")

## 3. Training

In [ ]:
model = AMLGraphAutoencoder(train_data.num_node_features, train_data.num_edge_features, config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
for epoch in range(101):
    optimizer.zero_grad()
    z, x_recon, edge_recon = model(train_data.x, train_data.edge_index, train_data.edge_attr)
    loss, _, _ = compute_combined_loss(train_data.x, x_recon, train_data.edge_attr, edge_recon, config)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0: print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

## 4. OOT Inference

In [ ]:
oot_data = builder.build_graph(oot_df)
oot_data.x = torch.from_numpy(node_scaler.transform(oot_data.x.numpy())).float().to(device)
oot_data.edge_attr = torch.from_numpy(edge_scaler.transform(oot_data.edge_attr.numpy())).float().to(device)
oot_data.edge_index = oot_data.edge_index.to(device)

model.eval()
with torch.no_grad():
    _, x_recon_oot, edge_recon_oot = model(oot_data.x, oot_data.edge_index, oot_data.edge_attr)
    node_errors = torch.mean((oot_data.x - x_recon_oot)**2, dim=1)
    edge_errors = torch.mean((oot_data.edge_attr - edge_recon_oot)**2, dim=1)

print("Inference complete.")

## 5. Global Explanations

In [ ]:
interpreter = AnomalyInterpreter(model, config)
print("\n--- Global Node Feature Importance ---")
interpreter.explain_node_anomalies(oot_data.x, x_recon_oot)

print("\n--- Global Edge Feature Importance ---")
interpreter.explain_edge_anomalies(oot_data.edge_attr, edge_recon_oot)

## 6. Interpretation & Export

In [ ]:
node_df, edge_df = interpreter.save_anomalies_to_excel(
    oot_data, node_errors, edge_errors, oot_df, '../output/anomalies.xlsx', 
    inv_map=builder.inv_customer_map,
    node_top_percent=0.05, 
    edge_top_percent=0.05
)

## 7. Local Perspectives

In [ ]:
print("\n--- Local Perspectives for Top 5 Node Anomalies ---")
top_node_indices = node_df.head(5)['node_idx'].tolist()
for idx in top_node_indices:
    cust_id = builder.inv_customer_map.get(int(idx))
    interpreter.local_perspective(
        int(idx), oot_data, node_errors, edge_errors, 
        num_hops=1, inv_map=builder.inv_customer_map,
        min_edge_risk_quantile=MIN_EDGE_RISK_QUANTILE
    )

print("\n--- Local Perspectives for Top 5 Edge Anomalies (Receiving Party) ---")
top_edge_targets = edge_df.head(5)['target'].tolist()
for idx in top_edge_targets:
    cust_id = builder.inv_customer_map.get(int(idx))
    interpreter.local_perspective(
        int(idx), oot_data, node_errors, edge_errors, 
        num_hops=1, inv_map=builder.inv_customer_map,
        min_edge_risk_quantile=MIN_EDGE_RISK_QUANTILE
    )